# TinyML Lab 5 - Part B: Anomaly Detection with Quantized Autoencoder on Arduino Nano BLE Sense

## Overview

In this lab, we deploy a quantized **autoencoder-based anomaly detection model** onto the **Arduino Nano 33 BLE Sense** board. Using real-time accelerometer data, the model classifies whether observed activity is *normal* or an *anomaly* based on reconstruction error. This experiment highlights the end-to-end TinyML workflow—from model training, quantization (INT8), and evaluation, to deployment on embedded hardware.

### Objectives

- Collect and preprocess motion sensor data (IMU) from Arduino Nano BLE.
- Train and evaluate a lightweight autoencoder for anomaly detection.
- Quantize the model using full integer quantization (INT8).
- Convert the model to a C array and deploy it to the Arduino board.
- Run inference in real-time and compute reconstruction error on-device.
- Detect anomalous patterns and display results via serial monitor.

### Key Concepts

- **Autoencoder**: A neural network trained to compress and reconstruct input data. Reconstruction error helps flag unseen anomalies.
- **Quantization (INT8)**: Reduces model size and latency by converting weights and activations to 8-bit integers.
- **TinyML Deployment**: Uses TensorFlow Lite for Microcontrollers (TFLM) to run inference on-device without OS support.

### Hardware Requirements

- Arduino Nano 33 BLE Sense
- USB cable
- Arduino IDE with necessary libraries:
  - `Arduino_LSM9DS1`
  - `TensorFlowLite` (TFLM)

---

Once complete, this lab demonstrates how to build an efficient, real-time anomaly detection system suitable for low-power, resource-constrained environments—a key application area in the TinyML ecosystem.



In [ ]:
# Uninstall `tensorflow` and `tensorflow-model-optimization` on Google Colab, due to API compatibility
#%pip uninstall -y keras tensorflow tensorflow-model-optimization
# Install `tensorflow` and `tensorflow-model-optimization`
%pip install "tensorflow<2.21" tensorflow-model-optimization tf_keras

# Install all required packages for this lab
%pip install numpy scipy scikit-learn pandas matplotlib umap-learn
%pip install seaborn


In [ ]:
import os

# Apply workaround before imports
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["KERAS_BACKEND"] = "tensorflow"

## 🧩 Section 1: Preprocessing and Windowing

In this section, we process raw time-series data from the **mHealth dataset** to prepare it for machine learning. Specifically, we focus on the following steps:

### ✅ Dataset Description
We use data from `mHealth_subject6.log`, which contains multi-sensor measurements during various human activities. Each row corresponds to a timestamped sensor reading, and the final column indicates the activity label.

- **Sensor columns used:** Left ankle accelerometer (columns 6, 7, 8 → 0-indexed as 5, 6, 7)
- **Label column:** Column 24 (0-indexed as 23)

### 🚫 Step 1: Remove Label 0
We exclude rows where the label is 0, which corresponds to "no activity recorded".

### ✂️ Step 2: Extract Relevant Features and Labels
We extract a 3-dimensional time-series using only the ankle acceleration values.

### 🔀 Step 3: Windowing
The dataset is divided into overlapping windows:
- **Window size:** 100 time steps
- **Stride:** 1 step
- Each window is flattened into a 1D array of shape `(3 × 100 = 300,)`.

### 📚 Step 4: Train/Test Split (Per Class)
For each activity label:
- We split the data **temporally**, keeping 70% of the sequence for training and 30% for testing.
- We then apply windowing separately for the training and testing sets.
- All windows and labels are accumulated into final datasets: `X_train`, `X_test`, `y_train`, and `y_test`.

This section prepares the dataset in a supervised format suitable for classification and unsupervised representation learning.


In [ ]:
import pandas as pd
import numpy as np

# Parameters for windowing
WINDOW_SIZE = 100
STRIDE = 1

# Load dataset
df = pd.read_csv("health-dataset/mHealth_subject6.log", sep='\t', header=None)

# Exclude unlabeled (label=0) entries
df = df[df[23] > 0]

# Extract ankle acceleration features and adjust label indexing
X_all = df[[5, 6, 7]].values  # Columns 6, 7, 8
y_all = df[23].values - 1     # Label column (0-based indexing)

# Containers for final dataset
X_train_windows = []
X_test_windows = []
y_train_labels = []
y_test_labels = []

# Segment data per class
for label in sorted(np.unique(y_all)):
    X_class = X_all[y_all == label]
    y_class = y_all[y_all == label]

    # Temporal split
    split_idx = int(0.7 * len(X_class))
    X_train_seq = X_class[:split_idx]
    X_test_seq = X_class[split_idx:]

    # Windowing function
    def create_windows(X_seq, label):
        windows = []
        labels = []
        for i in range(0, len(X_seq) - WINDOW_SIZE + 1, STRIDE):
            window = X_seq[i:i+WINDOW_SIZE].flatten()
            windows.append(window)
            labels.append(label)
        return windows, labels

    # Create windows and labels
    train_windows, train_labels = create_windows(X_train_seq, label)
    test_windows, test_labels = create_windows(X_test_seq, label)

    # Append to full dataset
    X_train_windows.extend(train_windows)
    y_train_labels.extend(train_labels)
    X_test_windows.extend(test_windows)
    y_test_labels.extend(test_labels)

# Final arrays
X_train = np.array(X_train_windows)
X_test = np.array(X_test_windows)
y_train = np.array(y_train_labels)
y_test = np.array(y_test_labels)

# Print shapes for sanity check
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


## 🌐 Section 2: Visualization with UMAP

In this section, we visualize the extracted time-series windows using **UMAP** (Uniform Manifold Approximation and Projection), a dimensionality reduction technique well-suited for visualizing high-dimensional data.

### 🔍 Objective
Understand the structure of sensor-based human activity windows in 2D space by projecting 300-dimensional vectors into 2 dimensions.

### 📦 Data Used
- We use the **test set (`X_test`)** from the previous section.
- Labels are used for coloring and interpreting clusters, but not for fitting UMAP.

### 🧪 Preprocessing
No scaling or normalization is applied at this stage, but can be added for improved UMAP performance.

### 🎨 Visualization
Each activity label is plotted with a unique color. We use:
- **`matplotlib`** for plotting.
- **`tab20` colormap** for consistent color assignment across activity classes.
- A legend showing activity names.

### 📌 Outcome
This step helps visualize whether time-series representations are naturally separable by activity class — which is useful to:
- Interpret data quality
- Evaluate feature representation
- Identify noisy or overlapping classes


In [ ]:
import matplotlib.pyplot as plt
import umap.umap_ as umap
import numpy as np

# Use test data for visualization
X_use = X_test

# UMAP projection to 2D
umap_model = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
X_train_umap = umap_model.fit_transform(X_use)

# Activity labels (0-based)
label_map = {
    0: "Standing still",
    1: "Sitting and relaxing",
    2: "Lying down",
    3: "Walking",
    4: "Climbing stairs",
    5: "Waist bends forward",
    6: "Frontal elevation of arms",
    7: "Knees bending (crouching)",
    8: "Cycling",
    9: "Jogging",
    10: "Running",
    11: "Jump front & back"
}

# Assign colors per label
unique_labels = sorted(np.unique(y_train))
cmap = plt.get_cmap('tab20')
colors = {label: cmap(i % 20) for i, label in enumerate(unique_labels)}

# Plot UMAP projection
plt.figure(figsize=(12, 6))
for label in unique_labels:
    idx = y_test == label
    plt.scatter(X_train_umap[idx, 0], X_train_umap[idx, 1], s=5, color=colors[label], label=label_map[label])

plt.title("UMAP Projection of Test Windows (Labeled by Activity)")
plt.xlabel("UMAP Component 1")
plt.ylabel("UMAP Component 2")
plt.legend(markerscale=3, fontsize=9, title="Activity", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()


## PCA Projection of Test Set Windows

Principal Component Analysis (PCA) is a linear dimensionality reduction technique that projects high-dimensional data into a lower-dimensional space by maximizing variance along orthogonal components. This helps visualize how activity clusters behave in terms of separability using only linear projections.

In this section, we apply PCA to reduce the 300-dimensional IMU feature vectors (100 timesteps × 3 axes) to 2 dimensions and color them according to activity labels.


In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

# Use test data for visualization
X_use = X_test

# PCA projection to 2D
pca_model = PCA(n_components=2)
X_pca = pca_model.fit_transform(X_use)

# Assign activity label names and colors
label_map = {
    0: "Standing still",
    1: "Sitting and relaxing",
    2: "Lying down",
    3: "Walking",
    4: "Climbing stairs",
    5: "Waist bends forward",
    6: "Frontal elevation of arms",
    7: "Knees bending (crouching)",
    8: "Cycling",
    9: "Jogging",
    10: "Running",
    11: "Jump front & back"
}
unique_labels = sorted(np.unique(y_test))
cmap = plt.get_cmap('tab20')
colors = {label: cmap(i % 20) for i, label in enumerate(unique_labels)}

# Plot PCA results
plt.figure(figsize=(12, 6))
for label in unique_labels:
    idx = y_test == label
    plt.scatter(X_pca[idx, 0], X_pca[idx, 1], s=5, color=colors[label], label=label_map[label])

plt.title("PCA Projection of Test Windows (Labeled by Activity)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(markerscale=3, fontsize=9, title="Activity", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()


## t-SNE Projection of Test Set Windows

t-distributed Stochastic Neighbor Embedding (t-SNE) is a nonlinear technique well-suited for visualizing high-dimensional datasets in 2D or 3D. It preserves local similarity structures, making it ideal for exploring cluster formation among activities.

We use the t-SNE method to project IMU feature vectors from the test set and examine how clearly distinct activities form clusters in the latent space.


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Reduce to 2D using t-SNE (note: can be slow)
X_tsne = TSNE(n_components=2, perplexity=30, learning_rate=300, random_state=42).fit_transform(X_use)

# Plot t-SNE results
plt.figure(figsize=(12, 6))
for label in unique_labels:
    idx = y_test == label
    plt.scatter(X_tsne[idx, 0], X_tsne[idx, 1], s=5, color=colors[label], label=label_map[label])

plt.title("t-SNE Projection of Test Windows (Labeled by Activity)")
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.legend(markerscale=3, fontsize=9, title="Activity", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()


## 🤖 Section 3: Autoencoder for Unsupervised Feature Learning

In this section, we use a **fully connected autoencoder** to learn a compressed representation (latent space) of the sensor data. This helps to:
- Capture intrinsic structure in the data.
- Reduce dimensionality.
- Visualize complex patterns more meaningfully in low dimensions using UMAP.

### 🧱 Architecture
- **Input Layer:** 300-dimensional flattened window.
- **Encoder:**
  - Dense(32, relu) → Dense(16, linear) to capture latent features.
- **Decoder:**
  - Dense(32, relu) → Dense(300, linear) to reconstruct the original input.
- **Latent Dimension:** 16 (can be changed to 8 or 32 depending on UMAP quality or model size).

### ⚙️ Training Details
- **Loss:** Mean Squared Error (MSE)
- **Optimizer:** Adam (learning rate = 0.005)
- **Epochs:** 100
- **Batch Size:** 256
- **Input:** `X_train` (windows from all classes)

### 🔍 Output
- The trained `autoencoder` model.
- The separate `encoder` model used to extract the latent features.
- A UMAP projection of the learned latent space (colored by activity labels).

This step forms the backbone of our anomaly detection strategy later, where the model will learn to reconstruct only "normal" activities.


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
import umap.umap_ as umap
import matplotlib.pyplot as plt
import numpy as np

# Define autoencoder architecture using Sequential API
input_dim = X_train.shape[1]
latent_dim = 16

# Create encoder and decoder layers for the autoencoder model
encoder_layers = [
    Dense(32, activation='relu', input_shape=(input_dim,)),
    Dense(latent_dim, activation='linear', name='latent')
]

encoder = Sequential(encoder_layers)
autoencoder = Sequential([
    *encoder_layers,
    # Decoder layers
    Dense(32, activation='relu'),
    Dense(input_dim, activation='linear')
])

# Compile and train
autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
autoencoder.fit(X_train, X_train, epochs=100, batch_size=256, shuffle=True, verbose=1)

# Extract latent features from test set
latent_features = encoder.predict(X_test)

# UMAP on latent space
umap_model = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
latent_umap = umap_model.fit_transform(latent_features)

# Plot
label_map = {
    0: "Standing still",
    1: "Sitting and relaxing",
    2: "Lying down",
    3: "Walking",
    4: "Climbing stairs",
    5: "Waist bends forward",
    6: "Frontal elevation of arms",
    7: "Knees bending (crouching)",
    8: "Cycling",
    9: "Jogging",
    10: "Running",
    11: "Jump front & back"
}
unique_labels = sorted(np.unique(y_train))
cmap = plt.get_cmap('tab20')
colors = {label: cmap(i % 20) for i, label in enumerate(unique_labels)}

plt.figure(figsize=(12, 6))
for label in unique_labels:
    idx = y_test == label
    plt.scatter(latent_umap[idx, 0], latent_umap[idx, 1], s=5, color=colors[label], label=label_map[label])

plt.title("UMAP of Autoencoder Latent Features (Test Set)")
plt.xlabel("UMAP Component 1")
plt.ylabel("UMAP Component 2")
plt.legend(markerscale=3, fontsize=9, title="Activity", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

### 🔍 PCA and t-SNE on Autoencoder Latent Feature Space

After compressing input data using an autoencoder, it’s insightful to visualize how different activities are separated in the **latent space** using PCA and t-SNE:

---

#### 📘 Principal Component Analysis (PCA)

PCA is a **linear** dimensionality reduction method that projects the data along the directions of maximum variance. It helps to:

- Visualize global trends in feature compression.  
- Check if the top two components carry enough variance to cluster activity types.

---

#### 📘 t-SNE (t-distributed Stochastic Neighbor Embedding)

t-SNE is a **non-linear** technique optimized for **local neighborhood preservation**. It's useful when:

- You want to observe **tight clusters**.  
- There may be **complex manifolds** in latent space that PCA can’t capture.

🔸 _Note:_ t-SNE is stochastic and sensitive to:

- `learning_rate`: Often needs to be set high (e.g., 200–1000) when data is compressed or high-dimensional.  
- `perplexity`: Balances attention between local/global clusters. Usually set between 5–50.


In [ ]:
# Perform PCA on latent features
pca_model = PCA(n_components=2)
latent_pca = pca_model.fit_transform(latent_features)

plt.figure(figsize=(12, 6))
for label in unique_labels:
    idx = y_test == label
    plt.scatter(latent_pca[idx, 0], latent_pca[idx, 1], s=5, color=colors[label], label=label_map[label])

plt.title("PCA of Autoencoder Latent Features (Test Set)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(markerscale=3, fontsize=9, title="Activity", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Perform t-SNE on latent features
tsne_model = TSNE(n_components=2, perplexity=30, learning_rate=300, random_state=42)
latent_tsne = tsne_model.fit_transform(latent_features)

plt.figure(figsize=(12, 6))
for label in unique_labels:
    idx = y_test == label
    plt.scatter(latent_tsne[idx, 0], latent_tsne[idx, 1], s=5, color=colors[label], label=label_map[label])

plt.title("t-SNE of Autoencoder Latent Features (Test Set)")
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.legend(markerscale=3, fontsize=9, title="Activity", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()


## 🚨 Section 4: Binary Classification for Anomaly Detection

This section prepares the dataset for anomaly detection using the previously trained autoencoder. The goal is to detect whether a given time-window represents a **normal activity** or an **anomalous activity**.

### 🎯 Motivation
Instead of classifying specific activities, we want to detect if a behavior is normal or suspicious (anomaly detection). This mimics real-world health/safety applications where unusual movement might signal danger or malfunction.

### 🧠 Binary Mapping
- We define a binary label function:
  - **Label 0 (Normal):** [Standing still, Sitting, Lying down, Bending, Frontal arm elevation, Knees bending]
  - **Label 1 (Anomaly):** All other active movements

### 🧪 Purpose
- We only **train the autoencoder on normal (label 0)** data so it learns to reconstruct normal movement.
- During testing, we expect high reconstruction errors for anomalies (label 1).

### 📋 Summary
- Convert the multi-class labels to binary.
- Split the dataset into:
  - `X_train_normal`: only normal data for training
  - `X_test_binary`, `y_test_binary`: for testing inference and performance


In [ ]:
# Define binary label mapping function
def to_binary_label(y):
    return np.where(np.isin(y, [0, 1, 2, 5, 6, 7]), 0, 1)

# Apply binary labeling
y_train_binary = to_binary_label(y_train)
y_test_binary = to_binary_label(y_test)

# Make binary-labeled datasets
X_train_binary = X_train.copy()
X_test_binary = X_test.copy()

# Print class distributions
print("Training class distribution:", np.bincount(y_train_binary))
print("Testing class distribution:", np.bincount(y_test_binary))


## 🔧 Section 5: Training Autoencoder Only on Normal Class for Anomaly Detection

### 🎯 Goal
Train the autoencoder **only on normal samples (label 0)** so it learns to accurately reconstruct typical behavior. During inference, any input with a high reconstruction error is considered **anomalous**.

### 🧠 Why This Works
- The autoencoder learns the structure of normal data.
- When it encounters anomalous inputs, it fails to reconstruct them accurately, resulting in higher reconstruction errors.

### 🧪 Steps Performed
1. **Subset Training Data:** Use only normal samples from the binary-labeled training set.
2. **Model Architecture:** A simple feedforward autoencoder with:
   - Input layer
   - Two hidden layers (encoder and decoder)
   - Linear latent layer
3. **Training Configuration:**
   - Loss: Mean Squared Error (MSE)
   - Optimizer: Adam with learning rate = 0.005
   - Epochs: 100
   - Batch size: 256

### 📈 Output
Once trained, the model is used to:
- Compute reconstruction error for both normal and anomalous samples.
- Derive threshold statistics (mean, std, min, max).


In [ ]:
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# Step 1: Filter normal (label=0) data for training
X_train_normal = X_train_binary[y_train_binary == 0]

# Step 2: Define and train autoencoder using Sequential API with shared layers
input_dim = X_train_normal.shape[1]
latent_dim = 16

# Create shared encoder and decoder layers
encoder_layers = [
    Dense(32, activation='relu', input_shape=(input_dim,)),
    Dense(latent_dim, activation='linear', name='latent')
]

encoder = Sequential(encoder_layers)
autoencoder = Sequential([
    *encoder_layers,
    # Decoder layers
    Dense(32, activation='relu'),
    Dense(input_dim, activation='linear')
])

autoencoder.compile(optimizer=Adam(learning_rate=0.005), loss='mse')
autoencoder.fit(X_train_normal, X_train_normal, epochs=100, batch_size=256, shuffle=True, verbose=1)

# Step 3: Compute reconstruction error statistics
def get_recon_stats(X):
    recon = autoencoder.predict(X)
    errors = np.mean((X - recon)**2, axis=1)
    mean_err = np.mean(errors)
    std_err = np.std(errors) / np.sqrt(len(errors))
    max_err = np.max(errors)
    min_err = np.min(errors)
    return mean_err, std_err, max_err, min_err

# Step 4: Evaluate on all four sets
mean_train0, stderr_train0, max_train0, min_train0 = get_recon_stats(X_train_binary[y_train_binary == 0])
mean_train1, stderr_train1, max_train1, min_train1 = get_recon_stats(X_train_binary[y_train_binary == 1])
mean_test0, stderr_test0, max_test0, min_test0 = get_recon_stats(X_test_binary[y_test_binary == 0])
mean_test1, stderr_test1, max_test1, min_test1 = get_recon_stats(X_test_binary[y_test_binary == 1])

# Step 5: Print results
print(f"Train Normal     (label 0): {mean_train0:.6f} ± {stderr_train0:.6f}; Max Error: {max_train0:.6f}; Min Error: {min_train0:.6f}")
print(f"Train Anomalous  (label 1): {mean_train1:.6f} ± {stderr_train1:.6f}; Max Error: {max_train1:.6f}; Min Error: {min_train1:.6f}")
print(f"Test Normal      (label 0): {mean_test0:.6f} ± {stderr_test0:.6f}; Max Error: {max_test0:.6f}; Min Error: {min_test0:.6f}")
print(f"Test Anomalous   (label 1): {mean_test1:.6f} ± {stderr_test1:.6f}; Max Error: {max_test1:.6f}; Min Error: {min_test1:.6f}")

## 📊 Section 6: Classification Using Reconstruction Error and Threshold

### 🎯 Goal
Use reconstruction error from the trained autoencoder to detect anomalies. Samples with high reconstruction errors are classified as anomalies.

### 🧠 How It Works
- The autoencoder is trained only on normal samples (label = 0).
- During inference:
  - Compute the Mean Squared Error (MSE) between input and reconstruction.
  - Apply a fixed threshold on the reconstruction error to classify:
    - **Error ≤ threshold → Normal**
    - **Error > threshold → Anomaly**

### 🛠️ Implementation Steps
1. **Compute Errors:**
   - Predict outputs for all test/train inputs.
   - Calculate MSE between input and reconstructed output.
2. **Apply Threshold:**
   - Choose a fixed threshold (e.g., 1.5) based on validation or visual inspection.
   - Classify each point as normal or anomalous.
3. **Evaluation:**
   - Generate classification reports (precision, recall, F1-score).
   - Visualize performance with confusion matrices.

### 📈 Output
You will see:
- Precision/Recall/F1 reports for both train and test sets.
- Confusion matrix visualizations to help assess model performance.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Step 1: Define function to get reconstruction errors
def get_recon_errors(X):
    recon = autoencoder.predict(X)
    return np.mean((X - recon) ** 2, axis=1)

# Step 2: Get reconstruction errors
train_errors = get_recon_errors(X_train_binary)
test_errors = get_recon_errors(X_test_binary)

# Step 3: Predict using a fixed threshold
threshold = 1.5
y_train_pred = (train_errors > threshold).astype(int)
y_test_pred = (test_errors > threshold).astype(int)

# Step 4: Print classification reports
print("🔎 Train Set Evaluation")
print(classification_report(y_train_binary, y_train_pred, target_names=["Normal (0)", "Anomaly (1)"]))

print("\n🔎 Test Set Evaluation")
print(classification_report(y_test_binary, y_test_pred, target_names=["Normal (0)", "Anomaly (1)"]))

# Step 5: Plot confusion matrices
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=["Normal", "Anomaly"],
                yticklabels=["Normal", "Anomaly"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(y_train_binary, y_train_pred, "Train Set Confusion Matrix")
plot_confusion_matrix(y_test_binary, y_test_pred, "Test Set Confusion Matrix")


## ⚙️ Section 7: Quantization-Aware Training and TFLite Conversion

### 🎯 Goal
Optimize the trained autoencoder for deployment on edge devices using Quantization-Aware Training (QAT), then convert the model to TensorFlow Lite (TFLite) format with full int8 support.

### 🧠 Why Quantization-Aware Training?
- Post-training quantization may degrade performance for small or sensitive models.
- QAT simulates quantization during training, allowing the model to adapt to reduced precision.
- Result: Smaller model size and faster inference, with minimal performance loss.

### 🛠️ Implementation Steps

1. **Apply QAT**  
   Use `tensorflow_model_optimization` to create a quantization-aware version of the trained autoencoder.

2. **Retrain the Model**  
   Fine-tune the QAT model with the same dataset used in normal training (only normal data).

3. **Convert to int8 TFLite Format**  
   Use a representative dataset and TFLite converter to generate a fully quantized model:
   - Inputs: `int8`
   - Outputs: `int8`
   - Weights and activations: `int8`

4. **Save the Model**
   Store the resulting `.tflite` file for deployment.

### 📦 Outcome
- You’ll obtain a compact `.tflite` file suitable for deployment on devices like Arduino Nano BLE 33.
- Accuracy remains high if QAT is properly tuned.


In [ ]:
import tensorflow_model_optimization as tfmot
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

# Step 1: Quantization-aware training setup
quantize_model = tfmot.quantization.keras.quantize_model
q_aware_autoencoder = quantize_model(autoencoder)

# Step 2: Compile and fine-tune the quantized model
q_aware_autoencoder.compile(optimizer=Adam(learning_rate=0.0001), loss='mse')
q_aware_autoencoder.fit(X_train_normal, X_train_normal, epochs=100, batch_size=256, shuffle=True, verbose=1)

# Step 3: Create representative dataset for TFLite conversion
def representative_dataset():
    for i in range(0, min(1000, len(X_train_normal)), 1):
        yield [X_train_normal[i:i+1].astype(np.float32)]

# Step 4: TFLite conversion with full int8 quantization
converter = tf.lite.TFLiteConverter.from_keras_model(q_aware_autoencoder)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# Step 5: Convert and save the model
tflite_model = converter.convert()
with open("autoencoder_int8.tflite", "wb") as f:
    f.write(tflite_model)


## 🤖 Section 8: Inference with the int8 Quantized Model (TFLite)

### 🎯 Goal
Run inference using the fully quantized int8 autoencoder model and compute the reconstruction error per sample. This is necessary to perform anomaly detection and validate the accuracy of the int8 model.

### ⚙️ Key Concepts

- **TFLite Interpreter**: Allows loading and running the quantized `.tflite` model in Python.
- **Quantization Parameters**:
  - **Scale and Zero-Point** are used to convert between float32 and int8 values.
  - Input: `x_int8 = round(x / scale + zero_point)`
  - Output: `y_float = (y_int8 - zero_point) * scale`

### 🧪 Evaluation Strategy
1. **Quantize input** → convert float input to int8 using input tensor's quantization scale and zero-point.
2. **Run inference** → feed input to the interpreter and invoke the model.
3. **Dequantize output** → convert int8 output back to float32.
4. **Compute reconstruction error** → Mean Squared Error (MSE) between original and reconstructed signal.
5. **Use a threshold** → flag high-error samples as anomalies.

### 📦 Outcome
- You will obtain per-sample reconstruction errors using int8 inference.
- These can be used to evaluate model performance on binary classification (normal vs. anomaly).


In [ ]:
import tensorflow as tf
import numpy as np

# Load the quantized TFLite model
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

# Get input/output tensor info
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

input_scale, input_zero_point = input_details[0]['quantization']
output_scale, output_zero_point = output_details[0]['quantization']

# Function to evaluate reconstruction error
def evaluate_quantized_recon_error(X):
    errors = []
    for x in X:
        # Quantize input
        x_int8 = np.round(x / input_scale + input_zero_point).astype(np.int8).reshape(1, -1)

        interpreter.set_tensor(input_details[0]['index'], x_int8)
        interpreter.invoke()

        # Dequantize output
        y_pred_int8 = interpreter.get_tensor(output_details[0]['index'])
        y_pred = (y_pred_int8.astype(np.float32) - output_zero_point) * output_scale

        # Reconstruction error
        error = np.mean((x - y_pred[0]) ** 2)
        errors.append(error)
    return np.array(errors)


## 📊 Section 9: Classification and Reporting of Results

### 🎯 Goal
Use the reconstruction errors from the int8 model to perform anomaly detection and evaluate performance using standard metrics like F1-score and confusion matrix.

### 📌 Key Steps

1. **Thresholding**:
   - Define a threshold on reconstruction error to distinguish between normal and anomalous data.
   - Common approach: If error > threshold → anomaly (label 1), else → normal (label 0).

2. **Classification Report**:
   - Use `classification_report` from `sklearn.metrics` to compute precision, recall, and F1-score.

3. **Confusion Matrix**:
   - Visualize true positives, false positives, etc., using a heatmap.
   - Helps identify the balance between detecting true anomalies and avoiding false alarms.

### 💡 Notes
- The threshold value (e.g., `1.5`) can be tuned based on the distribution of errors or validation data.
- Consistent labeling: 0 = Normal, 1 = Anomaly.

### 📦 Outcome
- Text-based classification reports (precision, recall, F1).
- Visual confusion matrices for both training and test sets.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Predict from reconstruction errors using threshold
threshold = 1.5

train_errors = evaluate_quantized_recon_error(X_train_binary)
test_errors = evaluate_quantized_recon_error(X_test_binary)

y_train_pred = (train_errors > threshold).astype(int)
y_test_pred = (test_errors > threshold).astype(int)

# Print classification reports
print("🔎 Train Set Evaluation")
print(classification_report(y_train_binary, y_train_pred, target_names=["Normal (0)", "Anomaly (1)"]))

print("\n🔎 Test Set Evaluation")
print(classification_report(y_test_binary, y_test_pred, target_names=["Normal (0)", "Anomaly (1)"]))

# Plot confusion matrix
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=["Normal", "Anomaly"],
                yticklabels=["Normal", "Anomaly"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(y_train_binary, y_train_pred, "Train Set Confusion Matrix")
plot_confusion_matrix(y_test_binary, y_test_pred, "Test Set Confusion Matrix")


## 🔹 Section 10: Memory & Size Estimation

### 🎯 Goal
Evaluate the resource requirements of the quantized and original autoencoder models, focusing on:
- Inference-time **RAM usage** (tensor memory allocation)
- **Model file size** for storage constraints (e.g., microcontroller flash)

### 📌 Key Steps

1. **RAM Usage Estimation (Tensor Memory)**:
   - Use `interpreter.get_tensor_details()` to gather tensor shapes and data types.
   - Estimate RAM usage by summing:  
     `tensor_memory = num_elements × bytes_per_element`.

2. **Model File Size Comparison**:
   - Save both the float32 and int8 `.tflite` models.
   - Use `os.path.getsize()` to compare their storage footprints.

3. **Optional**:
   - Display results in KB to understand deployability on embedded platforms (e.g., Arduino Nano BLE: 1MB flash, 256KB SRAM).

### 💡 Notes
- TFLite interpreter does not provide exact runtime memory profiling, but static tensor memory gives a reasonable approximation.
- File size reduction is one of the main benefits of quantization; int8 models are typically 4x smaller than float32 ones.

### 📦 Outcome
- Numeric estimates for memory and file size of both models.
- Confirms feasibility for deployment on memory-constrained edge devices.


In [ ]:
import numpy as np
import tensorflow as tf
import os

# Reload interpreter with int8 model
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

# Estimate memory used by tensors (RAM)
tensor_details = interpreter.get_tensor_details()
total_bytes = 0

for t in tensor_details:
    shape = t['shape']
    dtype = t['dtype']
    num_elements = np.prod(shape)
    bytes_per_element = np.dtype(dtype).itemsize
    total_bytes += num_elements * bytes_per_element

print(f"🧠 Estimated total memory for tensors: {total_bytes / 1024:.2f} KB")

# Save float32 model for comparison
converter = tf.lite.TFLiteConverter.from_keras_model(autoencoder)
tflite_model_fp32 = converter.convert()
with open("autoencoder.tflite", "wb") as f:
    f.write(tflite_model_fp32)

# Utility to get file size
def get_size_kb(path):
    size = os.path.getsize(path)
    return round(size / 1024, 2)

# Compare model file sizes
model_paths = {
    "Float32 (original)": "autoencoder.tflite",
    "Full Int8": "autoencoder_int8.tflite"
}

print("📦 TFLite Model Size Comparison:")
for name, path in model_paths.items():
    if os.path.exists(path):
        size_kb = get_size_kb(path)
        print(f" - {name}: {size_kb} KB")
    else:
        print(f" - {name}: file not found → {path}")


## 🔹 Section 11: C Array Export for Deployment

### 🎯 Goal
Convert the quantized TFLite model into a C array (`autoencoder_model.cc`) so it can be embedded directly into C++/Arduino firmware for deployment on microcontrollers (e.g., Arduino Nano BLE Sense).

### 📌 Key Steps

1. **Read Binary TFLite File**:
   - Open and read the quantized `.tflite` model as raw bytes.

2. **Format as C Header Array**:
   - Convert each byte to hexadecimal format (`0x00, ...`) and wrap into a `const unsigned char` array.
   - Include length variable for use in TensorFlow Lite Micro:
     ```cpp
     const unsigned char g_model[] = {...};
     const int g_model_len = ...;
     ```

3. **Save to File**:
   - The output `.cc` file (e.g., `autoencoder_model.cc`) can be added to Arduino sketches or PlatformIO projects directly.

### 💡 Notes
- This array is typically placed in **flash memory** (not RAM) on the device.
- TensorFlow Lite Micro uses this array via `tflite::GetModel(g_model)` during model initialization.

### 📦 Outcome
- Generates a ready-to-deploy C++ source file embedding the quantized autoencoder model for microcontroller inference.


In [ ]:
def convert_tflite_to_c_array(tflite_path, output_cc_path, array_name="g_model"):
    with open(tflite_path, "rb") as f:
        tflite_data = f.read()

    with open(output_cc_path, "w") as f:
        f.write(f"const unsigned char {array_name}[] = {{\n")

        for i, byte in enumerate(tflite_data):
            if i % 12 == 0:
                f.write("  ")
            f.write(f"0x{byte:02x}, ")
            if (i + 1) % 12 == 0:
                f.write("\n")

        f.write("\n};\n")
        f.write(f"const int {array_name}_len = {len(tflite_data)};\n")

# Convert and export C array for deployment
convert_tflite_to_c_array("autoencoder_int8.tflite", "autoencoder_model.cc")


## 🔹 Section 12: Deployment on Arduino Nano BLE – Real-Time Inference

### 🎯 Goal
Deploy the quantized autoencoder model on Arduino Nano BLE to detect anomalies in real-time from onboard accelerometer data.

---

### 🧩 Key Components Explained

#### ✅ 1. **Includes and Initialization**
- Load the TFLite model (`autoencoder_model.cc`) and necessary libraries.
- Allocate memory for inference with a fixed-size tensor arena.

#### ✅ 2. **Sensor Setup**
- Initializes the onboard IMU (accelerometer).
- A `100 × 3` buffer collects sensor readings in real-time.

#### ✅ 3. **Quantization Parameters**
- Automatically extracts model's quantization scale and zero-point for both input and output tensors to ensure proper quantization and dequantization.

#### ✅ 4. **Data Windowing and Preprocessing**
- Accelerometer readings are collected into a buffer.
- Once the window is filled, the buffer is flattened and quantized to match model expectations.

#### ✅ 5. **Inference and Reconstruction Error Calculation**
- The autoencoder reconstructs the input.
- The squared difference between original and reconstructed input is averaged to get a reconstruction error.

#### ✅ 6. **Anomaly Detection Logic**
- A threshold (e.g., 1.5) is applied.
- If the reconstruction error exceeds this threshold, an anomaly is flagged.

#### ✅ 7. **Loop Frequency**
- The delay of 20ms enforces a 50Hz sampling rate, matching the training data frequency.

---

### 💡 Notes
- Make sure the quantized model and `.cc` file are correctly placed in your Arduino project.
- This code is designed for **Arduino Nano BLE Sense**, but can be adapted to any MCU with compatible sensor and memory capacity.

---

### 📦 Outcome
- A standalone, deployable anomaly detection system on edge hardware using real sensor data, deep learning inference, and TFLite Micro.


```cpp
#include "TensorFlowLite.h"
#include "autoencoder_model.cc"  // Converted int8 TFLite model

#include "tensorflow/lite/micro/all_ops_resolver.h"
#include "tensorflow/lite/micro/micro_interpreter.h"
#include "tensorflow/lite/schema/schema_generated.h"
#include "tensorflow/lite/version.h"
#include "tensorflow/lite/micro/micro_error_reporter.h"
tflite::MicroErrorReporter micro_error_reporter;
tflite::ErrorReporter* error_reporter = &micro_error_reporter;

#include <Arduino_LSM9DS1.h>  // Load IMU Library

// Constants
const int kWindowSize = 100;
const int kInputSize = 300;  // 100 time steps × 3 axes
const int kTensorArenaSize = 64 * 1024;
uint8_t tensor_arena[kTensorArenaSize];

// Accelerometer buffer
float window_buffer[kWindowSize][3];
int sample_index = 0;
bool window_filled = false;

// TFLite variables
tflite::MicroInterpreter* interpreter;
TfLiteTensor* input;
TfLiteTensor* output;

// Quantization params (loaded after interpreter setup)
float input_scale;
int input_zero_point;
float output_scale;
int output_zero_point;

// Threshold
const float kReconstructionErrorThreshold = 1.5;

void setup() {
  Serial.begin(115200);
  while (!Serial);

  if (!IMU.begin()) {
    Serial.println("❌ Failed to initialize IMU!");
    while (1);
  }
  Serial.println("✅ IMU initialized");

  const tflite::Model* model = tflite::GetModel(g_model);
  static tflite::AllOpsResolver resolver;

  static tflite::MicroInterpreter static_interpreter(
    model, resolver, tensor_arena, kTensorArenaSize, error_reporter);
  interpreter = &static_interpreter;

  Serial.println("✅ Before AllocateTensors");
  if (interpreter->AllocateTensors() != kTfLiteOk) {
    Serial.println("❌ AllocateTensors() failed!");
    while (1);
  }
  Serial.println("✅ AllocateTensors successful");

  input = interpreter->input(0);
  output = interpreter->output(0);

  input_scale = input->params.scale;
  input_zero_point = input->params.zero_point;
  output_scale = output->params.scale;
  output_zero_point = output->params.zero_point;

  Serial.println("Model Setup Complete");
}

void loop() {
  float x, y, z;

  if (IMU.accelerationAvailable() && IMU.readAcceleration(x, y, z)) {
    window_buffer[sample_index][0] = x;
    window_buffer[sample_index][1] = y;
    window_buffer[sample_index][2] = z;
    sample_index++;

    if (sample_index >= kWindowSize) {
      sample_index = 0;
      window_filled = true;
    }

    if (window_filled) {
      // Quantize input
      for (int i = 0; i < kWindowSize; i++) {
        for (int j = 0; j < 3; j++) {
          float val = window_buffer[i][j];
          int index = i * 3 + j;
          int8_t quantized = static_cast<int8_t>(round(val / input_scale) + input_zero_point);
          input->data.int8[index] = quantized;
        }
      }

      // Inference
      if (interpreter->Invoke() != kTfLiteOk) {
        Serial.println("❌ Inference failed!");
        return;
      }

      // Reconstruction error
      float recon_error = 0.0;
      for (int i = 0; i < kInputSize; i++) {
        float original = window_buffer[i / 3][i % 3];
        int8_t quant_pred = output->data.int8[i];
        float predicted = (quant_pred - output_zero_point) * output_scale;
        recon_error += pow(original - predicted, 2);
      }
      recon_error /= kInputSize;

      // Display result
      Serial.print("Reconstruction error: ");
      Serial.println(recon_error, 6);

      if (recon_error > kReconstructionErrorThreshold) {
        Serial.println("🚨 Anomaly detected!");
      } else {
        Serial.println("✅ Normal activity");
      }
    }
  }

  delay(20);  // 50 Hz sampling
}
```




